# Ensembles: Bagging, Random Forests, and Boosting

In *Trees and Regularization*, a full tree had 0% training error and a poor test error: it was high variance. We fought that by making the tree smaller, with limits and pruning. This notebook fights it another way: grow **many** trees and combine them into an **ensemble** (ISL §8.2). Bagging and random forests average many deep trees. Boosting adds up many small trees, one after another.

### By the end, you'll be able to

9. Explain what it means for a single deep tree to be high variance.
10. Describe bagging ("Bootstrap Aggregating") in three steps, explain why averaging many trees reduces variance, and explain why adding more trees doesn't cause overfitting.
11. Explain out-of-bag (OOB) error and why it comes at no extra cost.
12. Describe how a random forest makes its trees less alike, explain the tradeoff in choosing $m$, and say when a random forest can fail.
13. Describe boosting for regression: fit a small tree to the current residuals, add a shrunken copy, repeat. Correct the common misconception about what each new tree is fit to.
14. Explain what boosting adds up for classification (a score, the log-odds) and what each new tree is fit to ($y - \hat p$).
15. Name each method's tuning parameters, say which method can overfit as trees are added, and contrast what each one mainly reduces: variance (bagging, random forests) or bias (boosting).

(The numbering continues from the *Trees and Regularization* notebook.)

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as pandas (`pd.`) and scikit-learn, do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_8_demos_helpers` draw a plot or build an interactive demo. You never need to open that file.

In [ ]:
import numpy as np
import pandas as pd

<a id="section-1"></a>

## 1 · The problem: one deep tree is high variance

In Demo C of *Trees and Regularization*, the full tree got every training point right and did much worse on new points, like KNN with $K = 1$. **High variance** means the fitted model depends a lot on which training data we happened to get: small changes in the training data produce very different trees.

We can see this directly. Below, we:
- make five slightly different versions of Chapter 5's 200 training points 
- grow a full tree on each. 

Each version is a **bootstrap sample**, the same tool you used in Chapter 5: 
- 200 rows drawn at random, with replacement, from the 200 training rows.

**Code worth reading.** `rng.integers(0, n, size=n)` picks $n$ row numbers at random, with replacement, as in Chapter 5. `tree.tree_.feature[0]` and `tree.tree_.threshold[0]` are the feature and split point of the tree's first split (scikit-learn numbers the features from 0).

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from ch_6_and_8_demos_helpers import simulate_data

sim = simulate_data()                       # Chapter 5's 200 training points, 10,000 test points
X_sim, y_sim = sim.X, sim.y
n = len(y_sim)
rng = np.random.default_rng(0)

trees, rows = [], []
for b in range(5):
    sample = rng.integers(0, n, size=n)     # a bootstrap sample: n row numbers, with replacement
    tree = DecisionTreeClassifier(random_state=0).fit(X_sim[sample], y_sim[sample])
    trees.append(tree)
    rows.append({'first split': f'X{tree.tree_.feature[0] + 1} <= {tree.tree_.threshold[0]:.2f}',
                 'leaves': tree.get_n_leaves(),
                 'test error': 1 - tree.score(sim.X_test, sim.y_test)})
pd.DataFrame(rows).round(3)

**Code worth reading.** How often do the five trees disagree about a new point? `predictions` has one row per tree and one column per test point.

In [ ]:
predictions = np.array([tree.predict(sim.X_test) for tree in trees])
# predictions != predictions[0]: compare every tree's row to tree 0's row (True where they differ).
# .any(axis=0): down each column, True if any tree differs from tree 0, i.e. the trees don't all agree on that point.
# .mean(): the fraction of test points where that's True (True counts as 1, False as 0).
disagree = (predictions != predictions[0]).any(axis=0).mean()
print(f'At least two of the five trees disagree on {disagree:.0%} of the 10,000 test points.')

- All five trees start with a split on $X_2$, but at different points, from $-0.42$ to $0.15$. 
- Below the first split they go their own ways: between 15 and 26 leaves each, and test errors from 15% to 21%. 
- Five nearly identical training sets, five different trees, and on about a quarter of new points they don't all agree. 
- **That's what high variance looks like.**

**Ask:** If you could only keep one of these five trees, how would you choose? Is there a better option than choosing?

<details>
<summary>Answer</summary>

You could pick the one with the lowest validation error, but that choice would itself depend on luck. 
- A better option is to **keep all five and let them vote**. 
- Where one tree makes a mistake that the others don't, the vote overrules it. That's the idea behind the next section.

</details>

<a id="section-2"></a>

## 2 · B'agging: average many trees

**Bagging** is short for **b**ootstrap **agg**regat**ing** (Breiman, 1996; ISL §8.2.1). Chapter 5 used the bootstrap to see how much an estimate would change with a different sample ([Chapter 5, Section 2](../ch_5/ch_5_demos_2_bootstrap.ipynb#section-2)). Bagging uses bootstrap samples to build many trees:

1. Draw $B$ bootstrap samples. Each is $n$ rows drawn **with replacement** from the $n$ training rows, so some rows appear more than once and some not at all.
2. On each sample, grow a deep, unpruned tree. Each tree deliberately overfits its own sample: **low bias, high variance**.
3. Aggregate: average the $B$ trees' predictions (regression), or take a majority vote (classification).

ISL writes the average as

$$
\hat f_{\text{bag}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat f^{*b}(x),
$$

where $\hat f^{*b}$ is the tree grown on bootstrap sample $b$.

**Why it works.** Remember Demo E in Chapter 5, where the average of 50 wild $K = 1$ boundaries landed close to the Bayes boundary? With real data we can't draw 50 new training sets. Bagging fakes it with bootstrap samples. Each tree is wild, but their mistakes point in different directions and partly cancel. **Averaging cuts variance without adding much bias**: each tree is still deep, so the average can still follow a complicated boundary.

<a id="demo-f"></a>

### Demo F: a bag of trees settles down

One `BaggingClassifier` with 300 trees, fit to Chapter 5's 200 training points. Slide $B$ to see the vote of the first $B$ trees.

In [ ]:
from ch_6_and_8_demos_helpers import bagging_demo

bagging_demo(sim, size_percent=80)

**Ask:** As $B$ grows from 1 to 300, does the test error keep falling, level off, or start rising?

<details>
<summary>Answer</summary>

It falls quickly at first and then **levels off**. Adding trees never makes bagging overfit: each new tree is just one more vote in the average, and once there are enough votes the average stops changing much. A few hundred trees is typical. More trees only cost time.

</details>

### Out-of-bag (OOB) error

A bootstrap sample of $n$ rows leaves out, on average, about a third of the $n$ training rows. The exact number varies a little from sample to sample; the optional [Going further](#section-7) section shows why it's about a third. Those left-out rows are that tree's **out-of-bag** rows: the tree never saw them.

- Every training row has a set of trees that never saw it, 
- That's about a third of the $B$ trees. 
- Let only those trees vote on that row, and compare the vote to the row's true class. 

Doing this for every row gives the **OOB error**: an estimate of test error, like a validation set's, at no extra cost. No rows were set aside, and nothing was refit.

**Code worth reading.** `oob_score=True` asks `BaggingClassifier` to compute it. `oob_score_` is the OOB accuracy, so the OOB error is `1 - oob_score_`.

In [ ]:
from sklearn.ensemble import BaggingClassifier

bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=300, oob_score=True,
                        random_state=0).fit(X_sim, y_sim)
print(f'OOB error:  {1 - bag.oob_score_:.1%}   (uses only the 200 training points)')
print(f'Test error: {1 - bag.score(sim.X_test, sim.y_test):.1%}   (10,000 new points)')

The OOB error (16.5%) is close to the test error on 10,000 new points (14.9%). It isn't exact: it's an estimate from only 200 rows, so it has some noise of its own. But we got it for free, from the training data alone.

<a id="section-3"></a>

## 3 · Random forests: make the trees less alike

**The limitation of bagging: Correlated Errors.** 

Suppose one feature dominates the split scoring method (RSS, Gini impurity, etc). Almost every bagged tree therefore splits on *that feature* at the root, and the trees end up similar. 

- Similar trees make similar mistakes.
- Their errors are **correlated**. 

That means knowing one tree's errors lets you predict the others' errors: at a point where one tree predicts too high, the other trees probably predict too high there as well, and where one is about right, so are the others. If the errors were uncorrelated, one tree's mistake would tell you nothing about the next tree's. 

- Averaging predictions does cancel **uncorrelated** mistakes (because they "point" in different directions)
- Averaging predictions can't cancel **correlated** mistakes (because they "point" in the same direction: all too high or all too low, for regression).

**The fix** (ISL §8.2.2). A **random forest** is bagging with one change: 

- At every split, the tree may only choose to split on one of a random subset of all possible features. 
- Notation: $m$ is the size of that subset and $p$ is the total number of features.
- A new subset of features is drawn at each split. 
- Each tree is still grown on its own bootstrap sample, so there are two sources of randomness:

    - **rows**, a new bootstrap sample for each tree, and
    - **candidate features**, a new random subset of $m$ features at each split.

**Choosing $m$.** For classification, ISL suggests $m \approx \sqrt{p}$. Setting $m = p$ lets every split consider every feature, which is plain bagging. In scikit-learn, $m$ is `max_features`, and the defaults differ:

| Model | Default `max_features` | Which means |
|---|---|---|
| `RandomForestClassifier` | `'sqrt'` | $m = \sqrt{p}$ (rounded down) |
| `RandomForestRegressor` | `1.0` | $m = p$: all features, so it's bagging |

So a `RandomForestRegressor` with default settings is bagging. To get a random forest for regression, set `max_features` yourself.

**The tradeoff.** 
- A smaller $m$ makes each tree somewhat worse: some splits can't use the best feature. 
- But it makes the trees' prediction errors less correlated. 

The trees are still grown deep, so strong features (very probably) still get used, just lower down or in a different order. And the extra errors that the randomness introduces are uncorrelated from tree to tree, so averaging cancels their effects. Usually the drop in correlation outweighs the cost, but not always, so **$m$ is a tuning parameter**.

**When it can fail.** With only a few features, there's little room to make the trees different. When most features are noise, a small random subset may contain no useful feature at all, and the split it forces is a poor one.

<a id="demo-g"></a>

### Demo G: how alike are the trees?

Let's test the argument directly. The data is a made-up regression problem with $p = 10$ features: feature 1 is strong, and features 2 through 10 are each moderately useful and somewhat correlated with each other and with feature 1. There are 300 training points and 1,000 test points.

For each $m$ from 1 to 10, we fit a 200-tree forest and look at each tree's **errors** on the test points: its prediction minus the true $y$, one number per test point. Then we record three numbers:

1. **Tree correlation:** the average correlation between two trees' error lists. High means the trees make the same mistakes.
2. **Single-tree MSE:** how good one tree is on its own, averaged over the trees.
3. **Forest MSE:** how good the whole forest's average is.

One data set gives a noisy answer, so we repeat everything for 5 different data sets and average.

**Code worth reading.** `forest.estimators_` is the list of the forest's fitted trees. `np.corrcoef` takes a list of error lists and returns their correlation matrix; the entries above the diagonal are the pairs of different trees. Expect this cell to take 10 to 20 seconds.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from ch_6_and_8_demos_helpers import forest_data


def tree_errors(forest, X_test, y_test):
    """One row per tree: that tree's prediction minus the truth, at each test point."""
    return np.array([tree.predict(X_test) - y_test for tree in forest.estimators_])


def average_correlation(errors):
    """The average correlation between two different trees' errors."""
    correlations = np.corrcoef(errors)
    pairs = np.triu_indices(len(errors), k=1)          # every pair of different trees, once
    return correlations[pairs].mean()


p = 10
rows = []
for seed in range(5):
    data = forest_data(seed=seed)
    for m in range(1, p + 1):
        forest = RandomForestRegressor(n_estimators=200, max_features=m, random_state=seed)
        forest.fit(data.X, data.y)
        errors = tree_errors(forest, data.X_test, data.y_test)
        rows.append({'seed': seed, 'm': m,
                     'tree correlation': average_correlation(errors),
                     'single-tree MSE': np.mean(errors ** 2),
                     'forest MSE': np.mean((forest.predict(data.X_test) - data.y_test) ** 2)})

results = pd.DataFrame(rows).groupby('m', as_index=False).mean().drop(columns='seed')
results.round(3)

In [ ]:
from ch_6_and_8_demos_helpers import plot_tree_correlation

plot_tree_correlation(results);

Read the panels from right to left, starting at $m = p = 10$, which is bagging.

- **Top:** as $m$ falls from 10 to 3, the trees' errors become less correlated, from about 0.31 to about 0.20.
- **Middle:** at the same time, each single tree gets worse, and much worse once $m$ is very small.
- **Bottom:** the forest improves at first, because the drop in correlation outweighs the worse trees, and it's best at $m = 4$ here. Below that, the worse trees win, and the forest gets much worse.

Notice the forest's MSE (about 6 to 7) is far below a single tree's (about 22 to 53): that's averaging at work, for every $m$.

At $m = 1$ and $2$, the correlation goes back **up**. With only one or two candidate features, most splits can't use feature 1, so every tree underestimates feature 1's effect in the same way. A mistake every tree shares is correlated error. This is the "when it can fail" case from above, and it's why $m$ is tuned rather than set as small as possible.

**Ask:** ISL suggests $m \approx \sqrt{p}$, which is about 3 here. How close was that to the best $m$ in this demo?

<details>
<summary>Answer</summary>

Close: the best was $m = 4$, and $m = 3$ was nearly as good (forest MSE about 6.5 vs. 6.0). $\sqrt{p}$ is a good starting point, but the best $m$ depends on the data, so it's worth tuning by cross-validation.

</details>

**Code worth reading.** Two heatmaps of one forest's tree-by-tree correlation matrix, for 30 trees: bagging ($m = p$) and a random forest ($m = 3$). Each square is one pair of trees; the brighter the square, the more alike their mistakes. The diagonal (each tree with itself) is left blank.

In [ ]:
from ch_6_and_8_demos_helpers import plot_correlation_heatmaps

data = forest_data(seed=0)
matrices = {}
for title, m in [('Bagging (m = p = 10)', 10), ('Random forest (m = 3)', 3)]:
    forest = RandomForestRegressor(n_estimators=30, max_features=m, random_state=0)
    forest.fit(data.X, data.y)
    matrices[title] = np.corrcoef(tree_errors(forest, data.X_test, data.y_test))
plot_correlation_heatmaps(matrices);

**Code worth reading.** A check that $m = p$ really is bagging: a random forest with `max_features=p` against `BaggingRegressor` with full trees, on the same data. They draw different random samples, so the numbers won't match exactly, but they should be close.

In [ ]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

data = forest_data(seed=0)
forest_m_equals_p = RandomForestRegressor(n_estimators=200, max_features=p, random_state=0)
bagged_trees = BaggingRegressor(DecisionTreeRegressor(), n_estimators=200, random_state=0)
for name, model in [('Random forest, m = p', forest_m_equals_p), ('Bagging', bagged_trees)]:
    model.fit(data.X, data.y)
    errors = tree_errors(model, data.X_test, data.y_test)
    test_mse = np.mean((model.predict(data.X_test) - data.y_test) ** 2)
    print(f'{name:22s} tree correlation {average_correlation(errors):.3f}   '
          f'forest test MSE {test_mse:.2f}')

**Feature importances.** A forest has hundreds of trees, so we can't read it the way we read one tree. Instead, scikit-learn measures how much each feature lowered the training error across all the splits that used it, and **averages over all the trees**. That's `feature_importances_`, scaled to add up to 1.

**Code worth reading.**

In [ ]:
forest = RandomForestRegressor(n_estimators=200, max_features=4, random_state=0).fit(data.X, data.y)
importances = pd.Series(forest.feature_importances_, index=[f'X{j}' for j in range(1, p + 1)])
importances.round(3)

Feature 1 has by far the largest importance, as it should: it's the strong feature in the recipe. The other nine share the rest.

**Ask:** You fit a random forest classifier with $p = 100$ features. What $m$ does ISL suggest, and what does scikit-learn use by default? What $m$ would turn it into bagging?

<details>
<summary>Answer</summary>

ISL suggests $m \approx \sqrt{100} = 10$, and `RandomForestClassifier`'s default `max_features='sqrt'` gives exactly that. Setting `max_features=100` (or `1.0`, meaning all of them) gives $m = p$: bagging.

</details>

<a id="section-4"></a>

## 4 · Boosting: many small corrections

Bagging and random forests grow deep trees independently and average them. **Boosting** takes a different approach (ISL §8.2.3): build **small** trees one after another, each one correcting the errors the model still makes. Here is gradient boosting for regression, following ISL's Algorithm 8.2.

1. Start with a model that predicts 0 everywhere: $\hat f(x) = 0$. So the residuals equal the $y$ values: $r_i = y_i$ for every training row $i$.
2. Repeat for $b = 1, 2, \dots, B$:
    1. Fit a **small** tree $\hat f^b$ (often only a few splits, sometimes just one) to the current residuals, using **all** the training rows. The features are the usual $X$; the target is $r$, not $y$.
    2. Add a shrunken copy of that tree to the model: $\hat f(x) \leftarrow \hat f(x) + \eta\, \hat f^b(x)$. The **learning rate** $\eta$ is small, such as $0.01$.
    3. Update the residuals: $r_i \leftarrow r_i - \eta\, \hat f^b(x_i)$.
3. The final model is the sum of all the shrunken trees: $\hat f(x) = \sum_{b=1}^{B} \eta\, \hat f^b(x)$.

> **A note on symbols.** ISL calls the learning rate $\lambda$ and names it the *shrinkage parameter* (equation 8.10). In this course we write $\eta$, and scikit-learn calls it `learning_rate`.

scikit-learn starts from the average of $y$ instead of 0, which saves the first few trees the work of finding the average. The idea is the same.

> **A common misconception.** You may hear boosting described as "each new tree focuses on the points the last one got wrong." That picture can mislead in two ways:
>
> - Boosting does **not** grow deep trees. Each tree is small, sometimes a single split.
> - Each new tree is **not** fit only on the misclassified or badly predicted points. Every tree is fit on **all** the training rows. What changes from tree to tree is the **target**: the residuals left over by the model so far.
>
> (The "focus on the mistakes" picture comes from an older method, AdaBoost; see [Going further](#section-7).)

**Tuning parameters.** Boosting has three:

| Setting | ISL | Course | scikit-learn | Default in `GradientBoostingRegressor` |
|---|---|---|---|---|
| Number of trees | $B$ | $B$ | `n_estimators` | 100 |
| Learning rate | $\lambda$ | $\eta$ | `learning_rate` | 0.1 |
| Size of each tree | $d$ splits | | `max_depth` (or `max_leaf_nodes`) | `max_depth=3` |

Unlike bagging and random forests, **boosting can overfit as $B$ grows**: keep adding corrections and the model eventually starts fitting the noise in the training data. So $B$ is chosen by cross-validation, along with $\eta$ and the tree size.

**The core idea: learn slowly.** Each tree only fixes a small part of what's left, and the learning rate shrinks that further. Many small, cautious corrections add up to a flexible model, without any single tree having to be good.

<a id="demo-h"></a>

### Demo H: watching the corrections add up

One feature $x$, 100 training points from a smooth curve plus noise.

**Code worth reading.** `max_depth=1` makes every tree a single split (a *stump*), and `learning_rate=0.05` is $\eta$. `staged_predict` returns the model's predictions after 1 tree, after 2 trees, and so on up to $B = 1{,}000$; we keep four of those stages.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from ch_6_and_8_demos_helpers import boosting_data, plot_boosting_stages

wave = boosting_data()
boost = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.05, max_depth=1,
                                  random_state=0).fit(wave.x, wave.y)

grid = np.linspace(0, 1, 500).reshape(-1, 1)        # x values to draw the fit at
stages = {}
for B, predictions in enumerate(boost.staged_predict(grid), start=1):
    if B in (1, 10, 100, 1000):
        stages[B] = predictions
plot_boosting_stages(wave, stages);

After 1 tree the fit is nearly flat: one stump, shrunk by $\eta = 0.05$, barely moves the starting average. After 10 trees it has barely moved. After 100 it follows the overall shape, and after 1,000 it follows the curve closely. No single stump can draw a wave; a thousand small steps can.

**Ask:** If we cut the learning rate from $0.05$ to $0.01$, what would the 100-tree panel look like? How would you get back to a good fit?

<details>
<summary>Answer</summary>

Each correction would be five times smaller, so after 100 trees the fit would be closer to flat, more like the 10-tree panel. To get back to a good fit, use more trees. A smaller $\eta$ needs a larger $B$; that's why $B$ and $\eta$ are tuned together.

</details>

### Boosting for classification (Optional)

For classification, the model doesn't add up class labels. It adds up a **score**: the log-odds of the positive class.

For example, with circles as the positive class (class 1), say a group of points is ○ ○ ○ △ (3 circles, 1 triangle):

$$
p = P(\circ) = \frac{3}{4}, \qquad \text{odds} = \frac{p}{1 - p} = \frac{3/4}{1/4} = 3, \qquad \text{log-odds} = \ln 3 \approx 1.10.
$$

An even mix (○ ○ △ △) gives $\ln 1 = 0$, and more triangles than circles (○ △ △ △) gives $\ln \tfrac{1}{3} \approx -1.10$. Gradient boosting starts with this score computed from the whole training set, and each tree nudges it up or down.

This is the same quantity logistic regression models as a straight line. The  logistic function (we'll call it the sigmoid when we study neural networks) turns the score into a probability, as in logistic regression:

$$
\hat p(x) = \frac{1}{1 + e^{-\text{score}(x)}}.
$$

With a score of $\ln 3$: $\hat p = \frac{1}{1 + 1/3} = \frac{3}{4}$, back to the fraction of circles.

Each new tree is fit to

$$
y - \hat p,
$$

the true label (0 or 1) minus the current predicted probability, and its shrunken output is added to the score. For example, a positive point ($y = 1$) predicted at $\hat p = 0.3$ gets a residual of $1 - 0.3 = 0.7$: a big push up. A positive point already predicted at $\hat p = 0.95$ gets $0.05$: it needs almost nothing.

These residuals are the **gradients** of the log-loss: the direction that lowers the loss fastest. That's where the name *gradient boosting* comes from.

Well-tuned gradient boosting (for example, the XGBoost library) is often the strongest method on tabular data, but it's more sensitive to its tuning parameters than a random forest. scikit-learn's `HistGradientBoostingClassifier` is a fast version, and the one you'll likely use on Kaggle.

<a id="section-5"></a>

## 5 · All together on the simulated data

<a id="demo-i"></a>

### Demo I: ensembles on the simulated data

**Code worth reading.** Five classifiers fit to Chapter 5's 200 training points and scored on its 10,000 test points. `BaggingClassifier(DecisionTreeClassifier(), n_estimators=300)` is bagging with 300 full trees. `RandomForestClassifier` is a random forest of full trees. The two boosting models use scikit-learn's defaults.

In [ ]:
from sklearn.ensemble import (BaggingClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from ch_6_and_8_demos_helpers import bayes_error_rate

models = {'One full tree': DecisionTreeClassifier(random_state=0),
          'Bagging (300 trees)': BaggingClassifier(DecisionTreeClassifier(), n_estimators=300,
                                                   random_state=0),
          'Random forest (300 trees)': RandomForestClassifier(n_estimators=300, random_state=0),
          'Gradient boosting': GradientBoostingClassifier(random_state=0),
          'Histogram gradient boosting': HistGradientBoostingClassifier(random_state=0)}
rows = []
for name, model in models.items():
    model.fit(X_sim, y_sim)
    rows.append({'model': name,
                 'training error': 1 - model.score(X_sim, y_sim),
                 'test error': 1 - model.score(sim.X_test, sim.y_test)})
print(f'Bayes error rate: {bayes_error_rate(sim):.1%}')
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
from ch_6_and_8_demos_helpers import plot_ensemble_boundaries

shown = ['One full tree', 'Bagging (300 trees)', 'Random forest (300 trees)',
         'Histogram gradient boosting']
plot_ensemble_boundaries(sim, {name: models[name] for name in shown});

Every ensemble beats the single full tree on new data, even though most of them still get every training point right. Their boundaries are still made of boxes, but the boxes average out into a shape closer to the Bayes boundary.

With only two features, a random forest classifier's default $m = \sqrt{2}$, rounded down, is 1 feature per split, so it barely differs from bagging here. The idea matters more with many features, as in Demo G and in the lab.

**Ask:** Adding more trees to a random forest never makes it overfit more. Adding more trees to boosting can. Why the difference?

<details>
<summary>Answer</summary>

Each forest tree is fit independently, on its own bootstrap sample, so adding trees just makes the average steadier. Each boosting tree is fit to the residuals the trees before it left on the training data. Keep adding trees and boosting keeps chasing whatever is left, including the noise. So for boosting, the number of trees is a flexibility setting to choose by cross-validation; for a forest, more trees only costs time.

</details>

<a id="section-6"></a>

## 6 · Summary

- A single deep tree is **high variance**: small changes in the training data give very different trees.
- **Bagging** grows a deep tree on each of $B$ bootstrap samples and averages them (or takes a vote). Averaging cuts variance without adding much bias. More trees never cause overfitting; the error levels off. **OOB error** estimates test error for free, from the rows each tree never saw.
- **Random forests** are bagging plus a random subset of $m$ of the $p$ features at every split. The trees get a bit worse but much less alike, and averaging cancels errors that differ from tree to tree. $m = p$ is bagging; $m$ is a tuning parameter.
- **Boosting** builds small trees one after another. Each is fit to the current residuals on **all** the training rows, and a shrunken copy (learning rate $\eta$) is added in. For classification it adds up a log-odds score and fits $y - \hat p$. $B$, $\eta$, and tree size are tuned by cross-validation, because boosting **can** overfit as $B$ grows.
- **The contrast:** bagging and random forests reduce **variance** by averaging many deep trees. Boosting mainly reduces **bias**, by building up the fit step by step from many simple trees.

<a id="section-7"></a>

## 7 · Going further (optional)

This section is for the curious. Nothing here is required.

**Why about a third of the rows are out-of-bag.** A bootstrap sample makes $n$ draws, each picking one of the $n$ rows. A given row is missed by one draw with probability $1 - 1/n$, so it's missed by all $n$ draws with probability

$$
P(\text{a given row is never drawn}) = \left(1 - \frac{1}{n}\right)^{n} \to e^{-1} \approx 0.368 \quad \text{as } n \to \infty.
$$

So, on average, each bootstrap sample contains about $1 - 0.368 = 0.632$, roughly two thirds, of the distinct rows, and the remaining third is out-of-bag. The exact number varies a little from sample to sample: with $n = 200$, about 95% of bootstrap samples leave out between 65 and 82 rows (33% to 41%). Even for $n = 200$, $(1 - 1/200)^{200} \approx 0.367$.

**Smaller bootstrap samples.** scikit-learn's bagging and forest models have a `max_samples` option that draws fewer than $n$ rows for each tree. Smaller samples make each tree faster and the trees less alike, at the cost of each tree seeing less data.

**How far can averaging go?** Suppose each of $B$ trees has variance $\sigma^2$, and every pair of trees has correlation $\rho$. The variance of their average is

$$
\rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2.
$$

As $B$ grows, the second term vanishes, leaving a floor of $\rho\,\sigma^2$. Adding trees can't get below that floor. Only making the trees less correlated can, and that's exactly what a random forest does: it lowers the floor by lowering $\rho$.

You can check this roughly against Demo G, using each tree's test MSE for $\sigma^2$. For bagging ($m = 10$), $\rho\,\sigma^2 \approx 0.313 \times 21.8 \approx 6.8$, and the 200-tree forest's MSE was 6.84. For $m = 4$, $\rho\,\sigma^2 \approx 0.221 \times 27.4 \approx 6.1$, and the forest's MSE was 6.01. With 200 trees, the forests are already sitting on their floors, and $m = 4$ has the lower floor even though its trees are worse.

**AdaBoost.** An older boosting method, AdaBoost (Freund and Schapire, 1997), works differently from gradient boosting: after each tree, it gives more **weight** to the training points that tree misclassified, so the next tree pays more attention to them. That's where the popular "focus on the mistakes" picture of boosting comes from. Gradient boosting, the version in this notebook, instead fits each tree to the residuals on all the rows.

# Practice Questions

These questions cover bagging, out-of-bag error, random forests, and boosting. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- Bagging: grow a deep tree on each of $B$ bootstrap samples; average the predictions (regression) or take a vote (classification).
- Random forest: bagging, plus each split chooses from a random $m$ of the $p$ features. $m = p$ is bagging.
- Boosting: start at 0; repeat $B$ times: fit a small tree to the residuals on all rows, add $\eta$ times it, update the residuals.

### Question 1: Bagging

(a) Describe bagging in three steps.

(b) Why does averaging many trees give lower test error than one full tree? (See [Section 2](#section-2).)

<details>
<summary>Hint</summary>

Think back to Chapter 5's Demo E and the average of many $K = 1$ boundaries.

</details>

<details>
<summary>Answer</summary>

(a) **Draw $B$ bootstrap samples of the training data; grow a deep tree on each; average their predictions (or take a vote).**

(b) Each full tree is high variance: its mistakes depend on its particular sample. The trees' mistakes point in different directions and partly cancel when averaged, so the bag's predictions change much less from sample to sample. **Averaging reduces variance**, and since each tree is still deep, it doesn't add much bias.

</details>

### Question 2: Out-of-bag

A bagged model has $B = 300$ trees, trained on $n = 200$ rows.

(a) About how many of the 300 trees never saw a given training row?

(b) Why does the OOB error not require setting aside a validation set?

<details>
<summary>Hint</summary>

On average, about what fraction of the rows does one bootstrap sample leave out?

</details>

<details>
<summary>Answer</summary>

(a) **About 100.** Each bootstrap sample leaves out, on average, about a third of the rows, so each row is out-of-bag for about a third of the trees. The exact count varies a little from row to row.

(b) Each row is scored only by the trees that never saw it, so those predictions are honest, like predictions on new data. **Every row gets used for training (by some trees) and for evaluation (by the others)**, so no rows have to be set aside and nothing has to be refit.

</details>

### Question 3: More trees

Your bagged model's test error is 15.0% with $B = 300$ trees. A classmate suggests $B = 3{,}000$ "to make it more flexible, so it may overfit." Are they right? What will most likely happen to the test error?

<details>
<summary>Answer</summary>

**No.** Adding trees to bagging (or a random forest) doesn't make it more flexible; each new tree is one more vote in an average that has already settled down. The test error will stay **about the same**, around 15%. The only cost is time.

</details>

### Question 4: Choosing $m$

A data set has $p = 25$ features.

(a) What $m$ does ISL suggest for a random forest classifier? What does `RandomForestClassifier` use by default?

(b) A classmate fits `RandomForestRegressor()` with its default settings and says "I used a random forest." What did they actually fit, and how would you fix it?

<details>
<summary>Hint</summary>

Look up the two defaults in the table in [Section 3](#section-3).

</details>

<details>
<summary>Answer</summary>

(a) **$m \approx \sqrt{25} = 5$**, and the default `max_features='sqrt'` gives exactly 5.

(b) **Bagging.** `RandomForestRegressor`'s default is `max_features=1.0`, all $p$ features, so $m = p$. To get a random forest, set `max_features` to something smaller (for example `max_features=8` or `'sqrt'`), and tune it, since $m$ is a tuning parameter.

</details>

### Question 5: One strong feature

In a data set, one feature is far stronger than the rest.

(a) Why do the bagged trees end up making similar mistakes?

(b) Why can't averaging more bagged trees fix that?

(c) How does a random forest help, and what does it cost? (See [Demo G](#demo-g).)

<details>
<summary>Answer</summary>

(a) Almost every bagged tree splits on the strong feature (the one that minimizes the split scoring function, such as RSS or Gini impurity) at the root, so the trees have similar shapes and **correlated errors**.

(b) Averaging cancels mistakes that point in different directions. A mistake that every tree makes in the same direction is not canceled by averaging, no matter how many trees there are.

(c) Each split can only choose from $m$ random features, so many trees start with a different feature and **the trees become less alike**, therefore their errors are far less correlated. The cost: **each tree is a bit worse**, since some splits can't use the best feature. In Demo G, the decrease in correlation won until $m$ got very small.

</details>

### Question 6: What does each tree see?

True or false? Explain each.

(a) In gradient boosting, each new tree is fit only to the points the model currently gets wrong.

(b) In gradient boosting, each tree is usually deep, so it can fix the errors in one step.

(c) In gradient boosting, the features stay the same from tree to tree; the target changes.

<details>
<summary>Answer</summary>

(a) **False.** Each tree is fit on **all** the training rows. What it's fit to is the current residuals.

(b) **False.** Boosting uses **small** trees, often only a few splits or just one, and shrinks each by the learning rate $\eta$. It learns slowly, through many small corrections.

(c) **True.** The features $X$ are the same every time. The target is the residuals, which change after every tree.

</details>

### Question 7: Classification residuals

A boosted classifier's current predicted probability of the positive class is $\hat p$ for each of these training points. What residual does the next tree get for each?

(a) $y = 1$, $\hat p = 0.3$

(b) $y = 1$, $\hat p = 0.95$

(c) $y = 0$, $\hat p = 0.8$

<details>
<summary>Hint</summary>

The residual is $y - \hat p$.

</details>

<details>
<summary>Answer</summary>

(a) $1 - 0.3 = $ **$0.7$**: a big push toward the positive class.

(b) $1 - 0.95 = $ **$0.05$**: it's nearly right already, so it needs almost nothing.

(c) $0 - 0.8 = $ **$-0.8$**: a big push toward the negative class. The tree's shrunken output is added to the log-odds score, which then moves down.

</details>

### Question 8: Forest or boosting?

(a) Which can overfit as you add more trees: a random forest or boosting? Why? (See [Demo I](#demo-i).)

(b) How should you choose $B$ for each?

<details>
<summary>Answer</summary>

(a) **Boosting.** Each new boosting tree is fit to what the earlier trees left over on the training data, so with enough trees it chases noise. A forest's trees are fit independently, so more trees only steady the average.

(b) For a forest, use enough trees that the error has leveled off (a few hundred is typical); more is safe, just slower. For boosting, **choose $B$ by cross-validation**, together with $\eta$ and the tree size.

</details>

### Question 9: Variance or bias?

Which ensemble methods mainly reduce variance, and which mainly reduce bias? Connect each to the kind of tree it uses.

<details>
<summary>Answer</summary>

**Bagging and random forests reduce variance.** They average many deep trees, each low bias and high variance; averaging keeps the low bias and cancels much of the variance.

**Boosting mainly reduces bias.** It starts from small trees, each very simple (high bias), and builds up the fit step by step, each tree correcting what's left.

</details>